In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import xarray as xr
import numpy as np
import torch
import math
import glob
import sys
import os

MAIN_DIR = 'PATH-TO-PROJECT-DIR'
sys.path.append(os.path.join(MAIN_DIR, 'library'))
from atmocean.macros import ALL_DATA_VARS, MEAN_DATA_VARS, STD_DATA_VARS

In [ ]:
ocean_data = 'oras5'

In [ ]:
class Welford(object):
    """ Implements Welford's algorithm for computing a running mean
    and standard deviation as described at: 
        http://www.johndcook.com/standard_deviation.html

    can take single values or iterables

    Properties:
        mean    - returns the mean
        std     - returns the std
        meanfull- returns the mean and std of the mean

    Usage:
        >>> foo = Welford()
        >>> foo(range(100))
        >>> foo
        <Welford: 49.5 +- 29.0114919759>
        >>> foo([1]*1000)
        >>> foo
        <Welford: 5.40909090909 +- 16.4437417146>
        >>> foo.mean
        5.409090909090906
        >>> foo.std
        16.44374171455467
        >>> foo.meanfull
        (5.409090909090906, 0.4957974674244838)
    """
    
    def __init__(self,lst=None):
        self.k = 0
        self.M = 0
        self.S = 0
        
        self.__call__(lst)
    
    def update(self,x):
        if x is None:
            return
        self.k += 1
        newM = self.M + (x - self.M)*1./self.k
        newS = self.S + (x - self.M)*(x - newM)
        self.M, self.S = newM, newS
    
    def consume(self,lst):
        n = len(lst)
        lst = iter(lst)
        for f in tqdm(lst, total=n):
            x = np.load(f, mmap_mode='r+').astype(np.float64)
            self.update(x)
    
    def __call__(self,x):
        if hasattr(x,"__iter__"):
            self.consume(x)
        else:
            self.update(x)
    
    @property
    def mean(self):
        return self.M
    @property
    def meanfull(self):
        return self.mean, self.std/math.sqrt(self.k)
    @property
    def std(self):
        if self.k==1:
            return 0
        return math.sqrt(self.S/(self.k-1))
    def __repr__(self):
        return "<Welford: {} +- {}>".format(self.mean, self.std)

In [ ]:
years = np.arange(1993, 2017)
years

In [ ]:
filenames = []
for year in years:
    filenames += sorted(glob.glob(os.path.join(MAIN_DIR, f'data/datasets/all_1_deg-{ocean_data}/{year}*.npy')))
len(filenames)

In [ ]:
welford = Welford(filenames)

In [ ]:
list(np.nanmean(welford.mean, axis=(-2,-1)))

In [ ]:
std = np.sqrt(welford.S / (welford.k - 1))
list(np.nanmax(std, axis=(-2,-1)))